# Assessment 1: Analysing historical data with system performance

**Student ID:** 33602190

## 1. Spark setup

### 1.1 Create the Spark session

In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

master = "local[*]"
app_name = "Assessment1-AML"
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 11:28:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/10/03 11:28:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### 1.2 Execution environment

In [2]:
print("Execution mode:", sc.master)
print("Processing threads:", sc.defaultParallelism)
print("Driver memory:", sc.getConf().get("spark.driver.memory"))
print("Executor memory: shared driver JVM in local mode; no separate executor heap")

Execution mode: local[*]
Processing threads: 14
Driver memory: 4g
Executor memory: shared driver JVM in local mode; no separate executor heap


## 2. Load and validate the data

The two HI-Small CSV files are loaded in full for validation. The transaction schema assigns unique names to the sender and receiver columns in their original CSV order. Account identifiers remain strings; bank identifiers are numeric in both datasets. Malformed records cause loading to fail rather than being silently discarded. No rows are removed or sampled here.

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, TimestampType, LongType,
    StringType, DoubleType, IntegerType
)

transaction_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("from_bank", LongType(), True),
    StructField("sender_account", StringType(), True),
    StructField("to_bank", LongType(), True),
    StructField("receiver_account", StringType(), True),
    StructField("amount_received", DoubleType(), True),
    StructField("receiving_currency", StringType(), True),
    StructField("amount_paid", DoubleType(), True),
    StructField("payment_currency", StringType(), True),
    StructField("payment_format", StringType(), True),
    StructField("is_laundering", IntegerType(), True)
])

account_schema = StructType([
    StructField("bank_name", StringType(), True),
    StructField("bank_id", LongType(), True),
    StructField("account_number", StringType(), True),
    StructField("entity_id", StringType(), True),
    StructField("entity_name", StringType(), True)
])

transactions = (
    spark.read.schema(transaction_schema)
    .option("header", True)
    .option("enforceSchema", True)
    .option("timestampFormat", "yyyy/MM/dd HH:mm")
    .option("mode", "FAILFAST")
    .csv("data/HI-Small_Trans.csv")
)
accounts = (
    spark.read.schema(account_schema)
    .option("header", True)
    .option("enforceSchema", True)
    .option("mode", "FAILFAST")
    .csv("data/HI-Small_accounts.csv")
)

transaction_count = transactions.count()
account_count = accounts.count()
print(f"Transactions: {transaction_count:,} rows; {len(transactions.columns)} columns")
print(f"Accounts: {account_count:,} rows; {len(accounts.columns)} columns")
transactions.printSchema()
accounts.printSchema()

Transactions: 5,078,345 rows; 11 columns
Accounts: 518,581 rows; 5 columns
root
 |-- timestamp: timestamp (nullable = true)
 |-- from_bank: long (nullable = true)
 |-- sender_account: string (nullable = true)
 |-- to_bank: long (nullable = true)
 |-- receiver_account: string (nullable = true)
 |-- amount_received: double (nullable = true)
 |-- receiving_currency: string (nullable = true)
 |-- amount_paid: double (nullable = true)
 |-- payment_currency: string (nullable = true)
 |-- payment_format: string (nullable = true)
 |-- is_laundering: integer (nullable = true)

root
 |-- bank_name: string (nullable = true)
 |-- bank_id: long (nullable = true)
 |-- account_number: string (nullable = true)
 |-- entity_id: string (nullable = true)
 |-- entity_name: string (nullable = true)



### 2.1 Missing values and valid values

Check every field for nulls and string fields for blanks. Check that laundering labels are 0 or 1 and monetary amounts are finite and zero or greater. Amounts retain the proposed double type; monetary totals must be grouped by currency, and later result comparisons must allow for floating point precision.

In [4]:
missing_results = []
for dataset_name, dataframe in [("Transactions", transactions), ("Accounts", accounts)]:
    checks = []
    for field in dataframe.schema.fields:
        missing = F.col(field.name).isNull()
        if isinstance(field.dataType, StringType):
            missing = missing | (F.trim(F.col(field.name)) == "")
        checks.append(F.sum(F.when(missing, 1).otherwise(0)).alias(field.name))
    counts = dataframe.agg(*checks).first().asDict()
    missing_results.extend((dataset_name, column, count) for column, count in counts.items())

spark.createDataFrame(missing_results, ["dataset", "column", "missing_rows"]).show(16, truncate=False)

transactions.agg(
    F.sum(F.when(~F.col("is_laundering").isin(0, 1), 1).otherwise(0)).alias("invalid_labels"),
    *[
        F.sum(F.when(
            F.isnan(column) | (F.abs(F.col(column)) == float("inf")) | (F.col(column) < 0),
            1
        ).otherwise(0)).alias(f"invalid_{column}")
        for column in ["amount_paid", "amount_received"]
    ]
).show(truncate=False)

+------------+------------------+------------+
|dataset     |column            |missing_rows|
+------------+------------------+------------+
|Transactions|timestamp         |0           |
|Transactions|from_bank         |0           |
|Transactions|sender_account    |0           |
|Transactions|to_bank           |0           |
|Transactions|receiver_account  |0           |
|Transactions|amount_received   |0           |
|Transactions|receiving_currency|0           |
|Transactions|amount_paid       |0           |
|Transactions|payment_currency  |0           |
|Transactions|payment_format    |0           |
|Transactions|is_laundering     |0           |
|Accounts    |bank_name         |0           |
|Accounts    |bank_id           |0           |
|Accounts    |account_number    |0           |
|Accounts    |entity_id         |0           |
|Accounts    |entity_name       |0           |
+------------+------------------+------------+



+--------------+-------------------+-----------------------+
|invalid_labels|invalid_amount_paid|invalid_amount_received|
+--------------+-------------------+-----------------------+
|0             |0                  |0                      |
+--------------+-------------------+-----------------------+



### 2.2 Date coverage, labels and sender cardinality

The source timestamps have no timezone offset. UTC is used consistently to preserve their recorded clock times and define calendar dates; this does not establish the real timezone of the simulated transactions. Labels identify the simulated laundering recorded in the source.

In [5]:
print("Timestamp parsing timezone:", spark.conf.get("spark.sql.session.timeZone"))
transactions.agg(
    F.min("timestamp").alias("first_transaction"),
    F.max("timestamp").alias("last_transaction"),
    F.countDistinct("sender_account").alias("distinct_sender_accounts")
).show(truncate=False)

transactions.groupBy("is_laundering").count().withColumn(
    "percentage", F.round(F.col("count") / transaction_count * 100, 4)
).orderBy("is_laundering").show()

Timestamp parsing timezone: UTC


+-------------------+-------------------+------------------------+
|first_transaction  |last_transaction   |distinct_sender_accounts|
+-------------------+-------------------+------------------------+
|2022-09-01 00:00:00|2022-09-18 16:18:00|496995                  |
+-------------------+-------------------+------------------------+



+-------------+-------+----------+
|is_laundering|  count|percentage|
+-------------+-------+----------+
|            0|5073168|   99.8981|
|            1|   5177|    0.1019|
+-------------+-------+----------+



### 2.3 Validate account joins

Check uniqueness of the composite account key `(bank_id, account_number)` and count transaction rows with unmatched sender or receiver keys. Duplicate account keys could multiply transactions when joined; unmatched keys could cause an inner join to drop transactions.

In [6]:
duplicate_account_keys = (
    accounts.groupBy("bank_id", "account_number")
    .count().filter(F.col("count") > 1).count()
)
account_keys = accounts.select("bank_id", "account_number").distinct()

unmatched_senders = transactions.join(
    account_keys,
    (transactions.from_bank == account_keys.bank_id)
    & (transactions.sender_account == account_keys.account_number),
    "left_anti"
).count()
unmatched_receivers = transactions.join(
    account_keys,
    (transactions.to_bank == account_keys.bank_id)
    & (transactions.receiver_account == account_keys.account_number),
    "left_anti"
).count()

spark.createDataFrame([
    ("Duplicate account key groups", duplicate_account_keys),
    ("Transactions with unmatched sender", unmatched_senders),
    ("Transactions with unmatched receiver", unmatched_receivers)
], ["check", "count"]).show(truncate=False)

+------------------------------------+-----+
|check                               |count|
+------------------------------------+-----+
|Duplicate account key groups        |0    |
|Transactions with unmatched sender  |0    |
|Transactions with unmatched receiver|0    |
+------------------------------------+-----+



The files contain **5,078,345 transactions** and **518,581 account records**. All fields contain values, all laundering labels are valid, and all amounts are finite and zero or greater. Transaction timestamps run from **1 September 2022 00:00** to **18 September 2022 16:18**.

There are **496,995 distinct sender account identifiers**. **5,177 transactions (0.1019%)** are labelled as laundering, so the final query threshold must account for the small labelled population.

The composite account key is unique and all sender/receiver keys match. Joining on both bank and account identifiers can therefore preserve transaction counts. No data cleaning or row removal has been applied.

## 3. Business query design

### 3.1 Sender entity type and laundering

Compare the proportion of transactions labelled as laundering across sender entity types. Derive the type from the text before `#` in `entity_name`, then join the account details using both the originating bank and sender account. Each transaction contributes once, and the percentage uses all transactions for that entity type as its denominator. This summary covers the full dataset before the daily account filters.

In [7]:
sender_entity_lookup = accounts.select(
    "bank_id", "account_number",
    F.trim(F.regexp_replace("entity_name", r"\s*#.*$", "")).alias("sender_entity_type")
)

sender_entity_summary = (
    transactions.select("from_bank", "sender_account", "is_laundering").alias("t")
    .join(
        sender_entity_lookup.alias("a"),
        (F.col("t.from_bank") == F.col("a.bank_id"))
        & (F.col("t.sender_account") == F.col("a.account_number")),
        "inner"
    )
    .groupBy("sender_entity_type")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("is_laundering").alias("laundering_count")
    )
    .withColumn("laundering_ratio", F.col("laundering_count") / F.col("transaction_count"))
    .orderBy(F.desc("laundering_ratio"), "sender_entity_type")
)

sender_entity_summary.select(
    "sender_entity_type", "transaction_count", "laundering_count",
    F.round(F.col("laundering_ratio") * 100, 4).alias("laundering_percent")
).show(truncate=False)

+-------------------+-----------------+----------------+------------------+
|sender_entity_type |transaction_count|laundering_count|laundering_percent|
+-------------------+-----------------+----------------+------------------+
|Individual         |4323             |6               |0.1388            |
|Corporation        |1409470          |1783            |0.1265            |
|Partnership        |1781367          |1805            |0.1013            |
|Sole Proprietorship|1790830          |1552            |0.0867            |
|Country            |91852            |31              |0.0337            |
|Direct             |503              |0               |0.0               |
+-------------------+-----------------+----------------+------------------+



Individuals have the highest labelled proportion at 0.1388%, but this is based on only 6 labelled transactions out of 4,323. Partnerships have the largest labelled count at 1,805. The difference shows why both transaction counts and proportions matter when interpreting the results. These figures describe the supplied simulated labels and do not establish that one entity type is inherently more likely to launder money.

Entity type provides context for the account review results below.

### 3.2 Payment formats, currencies and transaction values

Summarise the payment formats and currencies before comparing account activity. Monetary totals are calculated within each payment currency. The table shows the ten combinations with the most transactions; all combinations remain in the analysis.

In [8]:
payment_summary = (
    transactions.groupBy("payment_format", "payment_currency")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount_paid").alias("total_amount_paid"),
        F.sum("is_laundering").alias("laundering_count")
    )
    .orderBy(F.desc("transaction_count"), "payment_format", "payment_currency")
)
payment_summary.select(
    "payment_format", "payment_currency", "transaction_count",
    F.format_number("total_amount_paid", 6).alias("total_amount_paid"), "laundering_count"
).show(10, truncate=False)

+--------------+----------------+-----------------+----------------------+----------------+
|payment_format|payment_currency|transaction_count|total_amount_paid     |laundering_count|
+--------------+----------------+-----------------+----------------------+----------------+
|Cheque        |US Dollar       |708957           |269,429,963,512.778930|120             |
|Credit Card   |US Dollar       |506535           |1,659,661,152.160002  |90              |
|Cheque        |Euro            |441177           |140,245,443,208.850400|91              |
|Credit Card   |Euro            |315559           |871,308,548.469998    |52              |
|ACH           |US Dollar       |240028           |255,433,310,827.590120|1663            |
|Cash          |US Dollar       |187884           |99,038,162,603.580110 |39              |
|Reinvestment  |US Dollar       |186391           |47,279,104,270.499960 |0               |
|Bitcoin       |Bitcoin         |146013           |3,039,526.417936      |56    

### 3.3 Selected question and rules

**Which accounts should be prioritised for daily review using ACH transaction currency, value and rapid onward transfer behaviour, and how does this compare with reviewing all ACH payments?**

Exploration selected two conditions. Flag an ACH transaction if its payment currency is Saudi Riyal and its amount paid is at least 10,000 but below 100,000, or if it is either leg of a qualifying rapid onward transfer. The currency condition is a finding in this simulated dataset, not a regulatory threshold.

For rapid transfers, match an outgoing payment to the same account's nearest strictly earlier positive receipt in the same currency, within 24 hours. Both payments must use ACH, the final recipient must differ from the original sender, and the outgoing amount must be within 20% of the received amount. Self transfers are excluded from transfer matching. A receipt can match several outgoing payments; this is a behavioural association and does not trace the ownership of funds. Equal minute receipts are resolved by the largest assigned source row identifier. Timestamps within the same minute cannot establish an order.

Source rows receive an identifier once before either implementation. The input is persisted so those identifiers remain stable for both queries and their checks. Identical source rows remain separate transactions. Both legs of qualifying transfers are flagged, and each source row contributes once even if several conditions match. These flags support retrospective review after the onward payment is observed.

Rules were selected using 1 to 7 September. Results for 8 to 10 September provide a later comparison; 11 September onwards is reported separately because the population becomes very small and entirely ACH. Transfer matching stays within each period to prevent later activity affecting earlier selections. This is an exploratory temporal comparison, not an untouched external test.

In [9]:
from pyspark.sql import Window
from pyspark import StorageLevel

DISCOVERY_END = "2022-09-08"
EVALUATION_END = "2022-09-11"
TRANSFER_SECONDS = 24 * 60 * 60
AMOUNT_TOLERANCE = 0.20
VALUE_CURRENCY = "Saudi Riyal"
MIN_AMOUNT = 10000.0
MAX_AMOUNT = 100000.0

def review_period():
    return (F.when(F.col("timestamp") < DISCOVERY_END, "Discovery")
            .when(F.col("timestamp") < EVALUATION_END, "Evaluation")
            .otherwise("Tail"))

transaction_input = (
    transactions.withColumn("row_id", F.monotonically_increasing_id())
    .persist(StorageLevel.MEMORY_AND_DISK)
)
assert transaction_input.count() == transaction_count
assert transaction_input.select("row_id").distinct().count() == transaction_count

### 3.4 Daily account priority and evaluation

An account is identified by originating bank and sender account. Aggregate flagged activity by date, bank, account and payment currency. Keep currency separate for monetary totals. The final output includes bank and entity details, flagged transaction count and value, distinct recipients, reason counts and the proportion carrying laundering labels.

Rank accounts within each date by total flagged transactions across currencies, then rapid transfer transaction count, both descending. Bank and account identifiers break remaining ties. All currency rows for the same account and date share its rank. This ordering is a simple review workload rule, not a learned risk score. Laundering labels and monetary totals do not determine priority, and no accounts are removed by a top three cutoff.

Compare the currency condition, rapid transfer condition and their union with all ACH payments. Precision is labelled flagged transactions divided by all flagged transactions. Coverage is captured labels divided by all laundering labels in that period, including other payment formats. Workload is flagged transactions divided by all transactions. Daily account ranking has not been independently validated as a detection model.

## 4. DataFrame implementation

### 4.1 Identify transactions for review

Create incoming and outgoing event views using only transaction attributes. The window partitions by period, bank, account and currency, then selects the latest receipt within the preceding 24 hours. Labels do not enter the matching or filtering logic.

In [10]:
def build_dataframe_flags(source):
    flow = (
        source.select(
            "row_id", "timestamp", "from_bank", "sender_account", "to_bank",
            "receiver_account", "amount_paid", "payment_currency",
            "amount_received", "receiving_currency", "payment_format"
        )
        .filter((F.col("from_bank") != F.col("to_bank"))
                | (F.col("sender_account") != F.col("receiver_account")))
        .withColumn("review_period", review_period())
        .withColumn("event_second", F.col("timestamp").cast("long"))
    )
    outgoing = flow.select(
        "row_id", "review_period", "event_second", "payment_format",
        F.lit("out").alias("direction"), F.col("from_bank").alias("bank"),
        F.col("sender_account").alias("account"), F.col("to_bank").alias("other_bank"),
        F.col("receiver_account").alias("other_account"),
        F.col("payment_currency").alias("currency"), F.col("amount_paid").alias("amount")
    )
    incoming = flow.select(
        "row_id", "review_period", "event_second", "payment_format",
        F.lit("in").alias("direction"), F.col("to_bank").alias("bank"),
        F.col("receiver_account").alias("account"), F.col("from_bank").alias("other_bank"),
        F.col("sender_account").alias("other_account"),
        F.col("receiving_currency").alias("currency"), F.col("amount_received").alias("amount")
    )
    receipt_window = (
        Window.partitionBy("review_period", "bank", "account", "currency")
        .orderBy("event_second").rangeBetween(-TRANSFER_SECONDS, -1)
    )
    receipt = F.when(F.col("direction") == "in", F.struct(
        "event_second", "row_id", "amount", "other_bank", "other_account", "payment_format"
    ))
    matched = (
        incoming.unionByName(outgoing).filter(F.col("amount") > 0)
        .withColumn("receipt", F.max(receipt).over(receipt_window))
        .filter((F.col("direction") == "out") & (F.col("payment_format") == "ACH")
                & (F.col("receipt.payment_format") == "ACH")
                & ((F.col("other_bank") != F.col("receipt.other_bank"))
                   | (F.col("other_account") != F.col("receipt.other_account")))
                & (F.abs(F.col("amount") - F.col("receipt.amount"))
                   / F.col("receipt.amount") <= AMOUNT_TOLERANCE + 1e-12))
    )
    transfer_flags = matched.select(
        F.explode(F.array("row_id", "receipt.row_id")).alias("row_id"),
        F.lit(0).alias("value_flag"), F.lit(1).alias("transfer_flag")
    )
    value_flags = source.filter(
        (F.col("payment_format") == "ACH") & (F.col("payment_currency") == VALUE_CURRENCY)
        & (F.col("amount_paid") >= MIN_AMOUNT) & (F.col("amount_paid") < MAX_AMOUNT)
    ).select("row_id", F.lit(1).alias("value_flag"), F.lit(0).alias("transfer_flag"))
    return (
        value_flags.unionByName(transfer_flags).groupBy("row_id")
        .agg(F.max("value_flag").alias("value_flag"),
             F.max("transfer_flag").alias("transfer_flag"))
    )

### 4.2 Summarise and rank accounts

Aggregate before adding descriptive account information. Accounts with at least one flagged transaction enter the review list. The priority window uses counts across currencies; the displayed amounts remain grouped by currency.

In [11]:
def summarise_dataframe_flags(source, account_data, flags):
    flagged = (
        source.join(F.broadcast(flags), "row_id", "inner")
        .withColumn("review_period", review_period())
        .withColumn("transaction_date", F.to_date("timestamp"))
    )
    daily = (
        flagged.groupBy("review_period", "transaction_date", "from_bank",
                        "sender_account", "payment_currency")
        .agg(
            F.count("*").alias("flagged_count"),
            F.sum("amount_paid").alias("flagged_amount_paid"),
            F.countDistinct(F.struct("to_bank", "receiver_account")).alias("recipient_count"),
            F.sum("value_flag").alias("value_count"),
            F.sum("transfer_flag").alias("transfer_count"),
            F.sum("is_laundering").alias("labelled_count")
        )
        .withColumn("labelled_ratio", F.col("labelled_count") / F.col("flagged_count"))
    )
    account_days = daily.groupBy("transaction_date", "from_bank", "sender_account").agg(
        F.sum("flagged_count").alias("account_day_flags"),
        F.sum("transfer_count").alias("account_day_transfers")
    )
    priority = Window.partitionBy("transaction_date").orderBy(
        F.desc("account_day_flags"), F.desc("account_day_transfers"), "from_bank", "sender_account"
    )
    ranked = daily.join(
        account_days.withColumn("priority_rank", F.row_number().over(priority)),
        ["transaction_date", "from_bank", "sender_account"]
    )
    return (
        account_data.alias("a").join(
            F.broadcast(ranked).alias("r"),
            (F.col("a.bank_id") == F.col("r.from_bank"))
            & (F.col("a.account_number") == F.col("r.sender_account")), "inner"
        )
        .select("r.*", "a.bank_name", "a.entity_id", "a.entity_name")
        .orderBy("transaction_date", "priority_rank", "payment_currency")
    )

def build_dataframe_query(source, account_data):
    return summarise_dataframe_flags(source, account_data, build_dataframe_flags(source))

review_flags = build_dataframe_flags(transaction_input).cache()
flagged_transaction_count = review_flags.count()
dataframe_result = summarise_dataframe_flags(transaction_input, accounts, review_flags).cache()
dataframe_result_count = dataframe_result.count()
print(f"Flagged source transactions: {flagged_transaction_count:,}")
print(f"Daily account and currency rows: {dataframe_result_count:,}")

Flagged source transactions: 4,159
Daily account and currency rows: 3,976


### 4.3 Daily review output

The following views show the first ten account and currency rows in date and priority order. The complete result remains in `dataframe_result`. A transaction meeting both conditions contributes once to `flagged_count`, but appears in both reason counts. Label counts describe flagged transactions only.

In [12]:
preview = dataframe_result.orderBy("transaction_date", "priority_rank", "payment_currency").limit(10)
preview.select(
    "transaction_date", "priority_rank", "from_bank", "sender_account", "payment_currency",
    "flagged_count", "value_count", "transfer_count", "labelled_count"
).show(10, truncate=False)
preview.select(
    "transaction_date", "priority_rank", "payment_currency", "bank_name", "entity_name",
    F.round("flagged_amount_paid", 6).alias("flagged_amount_paid"), "recipient_count"
).show(10, truncate=False)

+----------------+-------------+---------+--------------+----------------+-------------+-----------+--------------+--------------+
|transaction_date|priority_rank|from_bank|sender_account|payment_currency|flagged_count|value_count|transfer_count|labelled_count|
+----------------+-------------+---------+--------------+----------------+-------------+-----------+--------------+--------------+
|2022-09-01      |1            |121      |811F03630     |Saudi Riyal     |3            |3          |0             |0             |
|2022-09-01      |2            |223      |8119F8CC0     |Saudi Riyal     |3            |3          |0             |1             |
|2022-09-01      |3            |118      |811BF5540     |Saudi Riyal     |2            |2          |0             |0             |
|2022-09-01      |4            |119      |811B987D0     |Saudi Riyal     |2            |2          |0             |0             |
|2022-09-01      |5            |121      |8000E1AF0     |Saudi Riyal     |2        

+----------------+-------------+----------------+--------------------+--------------------------+-------------------+---------------+
|transaction_date|priority_rank|payment_currency|bank_name           |entity_name               |flagged_amount_paid|recipient_count|
+----------------+-------------+----------------+--------------------+--------------------------+-------------------+---------------+
|2022-09-01      |1            |Saudi Riyal     |Saudi Arabia Bank #6|Partnership #38298        |116860.33          |3              |
|2022-09-01      |2            |Saudi Riyal     |Saudi Arabia Bank #4|Sole Proprietorship #16903|114102.89          |3              |
|2022-09-01      |3            |Saudi Riyal     |Switzerland Bank #8 |Corporation #37866        |79763.09           |2              |
|2022-09-01      |4            |Saudi Riyal     |Israel Bank #6      |Sole Proprietorship #3755 |40191.81           |2              |
|2022-09-01      |5            |Saudi Riyal     |Saudi Arabia 

### 4.4 Compare with reviewing all ACH payments

The source input is reused for the baseline denominators. Each condition is evaluated using distinct source rows, including legitimate duplicate transactions in the original file. The small flagged result is broadcast into the full population; absent flags are zero. Separate discovery, evaluation and tail results avoid presenting the unusual tail as evidence of general effectiveness.

In [13]:
scored_transactions = (
    transaction_input.join(F.broadcast(review_flags), "row_id", "left")
    .fillna({"value_flag": 0, "transfer_flag": 0})
    .withColumn("review_period", review_period())
)
comparison_conditions = {
    "All ACH": F.col("payment_format") == "ACH",
    "Currency and value": F.col("value_flag") == 1,
    "Rapid onward transfer": F.col("transfer_flag") == 1,
    "Combined review": (F.col("value_flag") == 1) | (F.col("transfer_flag") == 1)
}
comparison_expressions = [F.count("*").alias("population"),
                          F.sum("is_laundering").alias("population_labels")]
for i, condition in enumerate(comparison_conditions.values()):
    comparison_expressions.extend([
        F.sum(F.when(condition, 1).otherwise(0)).alias(f"flagged_{i}"),
        F.sum(F.when(condition, F.col("is_laundering")).otherwise(0)).alias(f"labelled_{i}")
    ])
period_counts = scored_transactions.groupBy("review_period").agg(*comparison_expressions).collect()
comparison_rows = []
for row in period_counts:
    for i, approach in enumerate(comparison_conditions):
        flagged, labelled = row[f"flagged_{i}"], row[f"labelled_{i}"]
        comparison_rows.append((row.review_period, approach, flagged, labelled,
                               labelled / flagged if flagged else 0.0,
                               labelled / row.population_labels,
                               flagged / row.population))
comparison = spark.createDataFrame(comparison_rows, [
    "review_period", "approach", "flagged_transactions", "labelled_transactions",
    "precision", "coverage", "workload"
])
comparison.select(
    "review_period", "approach", "flagged_transactions", "labelled_transactions",
    *[F.round(F.col(c) * 100, 2).alias(c + "_percent") for c in ["precision", "coverage", "workload"]]
).orderBy("review_period", "approach").show(12, truncate=False)

evaluation = {r.approach: r for r in comparison.filter(F.col("review_period") == "Evaluation").collect()}
chosen = evaluation["Combined review"]
baseline = evaluation["All ACH"]
print(f"Evaluation precision: {chosen.precision:.2%}; coverage: {chosen.coverage:.2%}.")
print(f"Precision relative to all ACH: {chosen.precision / baseline.precision:.2f} times.")
print(f"Flagged transactions without labels: {chosen.flagged_transactions - chosen.labelled_transactions:,}.")
print(f"Share of laundering labels missed: {1 - chosen.coverage:.2%}.")

+-------------+---------------------+--------------------+---------------------+-----------------+----------------+----------------+
|review_period|approach             |flagged_transactions|labelled_transactions|precision_percent|coverage_percent|workload_percent|
+-------------+---------------------+--------------------+---------------------+-----------------+----------------+----------------+
|Discovery    |All ACH              |422055              |2529                 |0.6              |83.55           |11.31           |
|Discovery    |Combined review      |2783                |235                  |8.44             |7.76            |0.07            |
|Discovery    |Currency and value   |1554                |141                  |9.07             |4.66            |0.04            |
|Discovery    |Rapid onward transfer|1239                |104                  |8.39             |3.44            |0.03            |
|Evaluation   |All ACH              |177634              |1299       

## 5. Spark SQL implementation

Register the prepared transaction input and account reference as temporary views. SQL independently reconstructs the events, receipt window and deduplicated flags. It does not read the DataFrame flag result. Common Table Expressions separate matching, daily aggregation, account priority and enrichment.

In [14]:
transaction_input.createOrReplaceTempView("transaction_input")
accounts.createOrReplaceTempView("account_reference")

sql_flag_ctes = f"""
source AS (
    SELECT *, CAST(timestamp AS BIGINT) AS event_second,
           CASE WHEN timestamp < TIMESTAMP '{DISCOVERY_END}' THEN 'Discovery'
                WHEN timestamp < TIMESTAMP '{EVALUATION_END}' THEN 'Evaluation'
                ELSE 'Tail' END AS review_period
    FROM transaction_input
),
flow AS (
    SELECT * FROM source
    WHERE from_bank <> to_bank OR sender_account <> receiver_account
),
events AS (
    SELECT row_id, review_period, event_second, payment_format, 'out' AS direction,
           from_bank AS bank, sender_account AS account,
           to_bank AS other_bank, receiver_account AS other_account,
           payment_currency AS currency, amount_paid AS amount
    FROM flow WHERE amount_paid > 0
    UNION ALL
    SELECT row_id, review_period, event_second, payment_format, 'in' AS direction,
           to_bank AS bank, receiver_account AS account,
           from_bank AS other_bank, sender_account AS other_account,
           receiving_currency AS currency, amount_received AS amount
    FROM flow WHERE amount_received > 0
),
with_receipts AS (
    SELECT *, MAX(CASE WHEN direction = 'in' THEN
        named_struct('event_second', event_second, 'row_id', row_id, 'amount', amount,
                     'other_bank', other_bank, 'other_account', other_account,
                     'payment_format', payment_format) END)
        OVER (PARTITION BY review_period, bank, account, currency ORDER BY event_second
              RANGE BETWEEN {TRANSFER_SECONDS} PRECEDING AND 1 PRECEDING) AS receipt
    FROM events
),
matched AS (
    SELECT row_id, receipt.row_id AS receipt_id
    FROM with_receipts
    WHERE direction = 'out' AND payment_format = 'ACH' AND receipt.payment_format = 'ACH'
      AND (other_bank <> receipt.other_bank OR other_account <> receipt.other_account)
      AND ABS(amount - receipt.amount) / receipt.amount <= {AMOUNT_TOLERANCE + 1e-12}
),
flag_candidates AS (
    SELECT row_id, 1 AS value_flag, 0 AS transfer_flag
    FROM source
    WHERE payment_format = 'ACH' AND payment_currency = '{VALUE_CURRENCY}'
      AND amount_paid >= {MIN_AMOUNT} AND amount_paid < {MAX_AMOUNT}
    UNION ALL
    SELECT row_id, 0 AS value_flag, 1 AS transfer_flag FROM matched
    UNION ALL
    SELECT receipt_id AS row_id, 0 AS value_flag, 1 AS transfer_flag FROM matched
),
flags AS (
    SELECT row_id, MAX(value_flag) AS value_flag, MAX(transfer_flag) AS transfer_flag
    FROM flag_candidates GROUP BY row_id
)
"""

In [15]:
sql_account_ctes = """
, daily AS (
    SELECT s.review_period, TO_DATE(s.timestamp) AS transaction_date,
           s.from_bank, s.sender_account, s.payment_currency,
           COUNT(*) AS flagged_count, SUM(s.amount_paid) AS flagged_amount_paid,
           COUNT(DISTINCT named_struct('to_bank', s.to_bank,
                                      'receiver_account', s.receiver_account)) AS recipient_count,
           SUM(f.value_flag) AS value_count, SUM(f.transfer_flag) AS transfer_count,
           SUM(s.is_laundering) AS labelled_count,
           SUM(s.is_laundering) / COUNT(*) AS labelled_ratio
    FROM source s INNER JOIN flags f ON s.row_id = f.row_id
    GROUP BY s.review_period, TO_DATE(s.timestamp), s.from_bank,
             s.sender_account, s.payment_currency
),
account_days AS (
    SELECT transaction_date, from_bank, sender_account,
           SUM(flagged_count) AS account_day_flags,
           SUM(transfer_count) AS account_day_transfers
    FROM daily GROUP BY transaction_date, from_bank, sender_account
),
priorities AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY transaction_date
        ORDER BY account_day_flags DESC, account_day_transfers DESC, from_bank, sender_account
    ) AS priority_rank FROM account_days
),
ranked AS (
    SELECT d.*, p.account_day_flags, p.account_day_transfers, p.priority_rank
    FROM daily d INNER JOIN priorities p
      ON d.transaction_date = p.transaction_date AND d.from_bank = p.from_bank
     AND d.sender_account = p.sender_account
)
SELECT /*+ BROADCAST(r) */ r.*, a.bank_name, a.entity_id, a.entity_name
FROM account_reference a INNER JOIN ranked r
  ON a.bank_id = r.from_bank AND a.account_number = r.sender_account
ORDER BY r.transaction_date, r.priority_rank, r.payment_currency
"""

def build_sql_query():
    return spark.sql("WITH " + sql_flag_ctes + sql_account_ctes)

sql_result = build_sql_query().select(*dataframe_result.columns).cache()
sql_result_count = sql_result.count()
print(f"SQL daily account and currency rows: {sql_result_count:,}")
sql_result.select(
    "transaction_date", "priority_rank", "from_bank", "sender_account", "payment_currency",
    "flagged_count", "labelled_count"
).orderBy("transaction_date", "priority_rank", "payment_currency").show(10, truncate=False)

SQL daily account and currency rows: 3,976
+----------------+-------------+---------+--------------+----------------+-------------+--------------+
|transaction_date|priority_rank|from_bank|sender_account|payment_currency|flagged_count|labelled_count|
+----------------+-------------+---------+--------------+----------------+-------------+--------------+
|2022-09-01      |1            |121      |811F03630     |Saudi Riyal     |3            |0             |
|2022-09-01      |2            |223      |8119F8CC0     |Saudi Riyal     |3            |1             |
|2022-09-01      |3            |118      |811BF5540     |Saudi Riyal     |2            |0             |
|2022-09-01      |4            |119      |811B987D0     |Saudi Riyal     |2            |0             |
|2022-09-01      |5            |121      |8000E1AF0     |Saudi Riyal     |2            |0             |
|2022-09-01      |6            |121      |811A4E080     |Saudi Riyal     |2            |0             |
|2022-09-01      |7  

## 6. Result validation and interpretation

### 6.1 Compare every DataFrame and SQL result row

Compare all identifiers, counts, rankings and descriptive fields in both directions. Compare monetary sums separately using a small absolute or relative tolerance because aggregation order can change the last digits of double precision sums. Check the daily keys, rank uniqueness and source transaction totals. These checks cover the complete results, not only the previews.

In [16]:
result_keys = ["review_period", "transaction_date", "from_bank", "sender_account", "payment_currency"]
exact_columns = [c for c in dataframe_result.columns if c != "flagged_amount_paid"]
assert dataframe_result_count == sql_result_count
assert dataframe_result.select(*exact_columns).exceptAll(sql_result.select(*exact_columns)).count() == 0
assert sql_result.select(*exact_columns).exceptAll(dataframe_result.select(*exact_columns)).count() == 0
amount_checks = dataframe_result.alias("d").join(sql_result.alias("s"), result_keys).filter(
    F.abs(F.col("d.flagged_amount_paid") - F.col("s.flagged_amount_paid"))
    > F.greatest(F.lit(1e-8), F.abs(F.col("d.flagged_amount_paid")) * F.lit(1e-10))
).count()
assert amount_checks == 0
assert dataframe_result.groupBy(*result_keys).count().filter(F.col("count") != 1).count() == 0
assert dataframe_result.filter(
    (F.col("flagged_count") < 1) | (F.col("labelled_count") < 0)
    | (F.col("labelled_count") > F.col("flagged_count"))
    | (F.col("recipient_count") > F.col("flagged_count"))
    | (F.col("value_count") > F.col("flagged_count"))
    | (F.col("transfer_count") > F.col("flagged_count"))
).count() == 0
ranked_accounts = dataframe_result.select(
    "transaction_date", "from_bank", "sender_account", "priority_rank"
).distinct()
assert ranked_accounts.groupBy("transaction_date", "priority_rank").count().filter(F.col("count") != 1).count() == 0
result_totals = dataframe_result.agg(
    F.sum("flagged_count").alias("transactions"), F.sum("labelled_count").alias("labels"),
    F.countDistinct(F.struct("from_bank", "sender_account")).alias("accounts")
).first()
assert result_totals.transactions == flagged_transaction_count
expected_labels = sum(r.labelled_transactions for r in comparison.filter(F.col("approach") == "Combined review").collect())
assert result_totals.labels == expected_labels
print("All result rows agree; monetary sums are within tolerance.")
print(f"Distinct accounts: {result_totals.accounts:,}; source transactions: {result_totals.transactions:,}; labels: {result_totals.labels:,}.")

All result rows agree; monetary sums are within tolerance.
Distinct accounts: 2,300; source transactions: 4,159; labels: 462.


### 6.2 Check transfer boundaries and label independence

A small constructed input checks the exact 24 hour boundary, amount tolerance, equal timestamps, different currencies, self transfers, period boundaries, latest non ACH receipts and the currency value boundaries. Its labels are deliberately all zero; changing or removing labels must not change the flags. Both APIs must return the same expected source identifiers.

In [17]:
from datetime import datetime, timedelta, timezone

fixture_rows = []
expected_ids = set()
def add_pair(case, minutes, outgoing_amount=100.0, outgoing_currency="US Dollar", incoming_format="ACH", outgoing_format="ACH", start="2022-09-02 01:00", same_account=False):
    first_id = case * 10
    second_id = first_id + 1
    first_time = datetime.strptime(start, "%Y-%m-%d %H:%M").replace(tzinfo=timezone.utc)
    a, b, c = f"A{case}", f"B{case}", f"C{case}"
    fixture_rows.append((first_time, 1, b if same_account else a, 1, b, 100.0, "US Dollar", 100.0, "US Dollar", incoming_format, 0, first_id))
    fixture_rows.append((first_time + timedelta(minutes=minutes), 1, b, 1, c, outgoing_amount, outgoing_currency, outgoing_amount, outgoing_currency, outgoing_format, 0, second_id))
    return first_id, second_id

expected_ids.update(add_pair(1, 60, 120.0))
add_pair(2, 60, 120.01)
expected_ids.update(add_pair(3, 1440))
add_pair(4, 1441)
add_pair(5, 0)
add_pair(6, 60, outgoing_currency="Euro")
add_pair(7, 60, incoming_format="Wire")
add_pair(8, 60, same_account=True)
add_pair(9, 60, start="2022-09-07 23:30")
add_pair(10, 60)
fixture_rows.append((datetime(2022, 9, 2, 1, 30, tzinfo=timezone.utc), 1, "D10", 1, "B10", 100.0, "US Dollar", 100.0, "US Dollar", "Wire", 0, 102))
for row_id, amount in [(201, 10000.0), (202, 9999.99), (203, 100000.0)]:
    fixture_rows.append((datetime(2022, 9, 2, 1, tzinfo=timezone.utc), 2, "V" + str(row_id), 2, "Z" + str(row_id), amount, "Saudi Riyal", amount, "Saudi Riyal", "ACH", 0, row_id))
expected_ids.add(201)
fixture_schema = StructType(transaction_schema.fields + [StructField("row_id", LongType(), False)])
fixture = spark.createDataFrame(fixture_rows, fixture_schema)
fixture_flags = build_dataframe_flags(fixture.drop("is_laundering"))
actual_ids = {r.row_id for r in fixture_flags.collect()}
assert actual_ids == expected_ids, (actual_ids, expected_ids)
assert {r.row_id for r in build_dataframe_flags(fixture.withColumn("is_laundering", F.lit(1))).collect()} == expected_ids
try:
    fixture.createOrReplaceTempView("transaction_input")
    sql_fixture_flags = spark.sql("WITH " + sql_flag_ctes + " SELECT * FROM flags")
    assert fixture_flags.exceptAll(sql_fixture_flags.select(*fixture_flags.columns)).count() == 0
    assert sql_fixture_flags.select(*fixture_flags.columns).exceptAll(fixture_flags).count() == 0
finally:
    transaction_input.createOrReplaceTempView("transaction_input")
print("Boundary cases and label independence passed for both APIs.")

Boundary cases and label independence passed for both APIs.


### 6.3 Interpretation and query design

The complete result contains 3,976 daily account and currency rows across 2,300 accounts, covering 4,159 transactions and 462 laundering labels. On 8 to 10 September, 1,269 flagged transactions include 127 labels, giving 10.01% precision and 8.49% coverage. Reviewing all ACH payments gives 0.73% precision and 86.89% coverage. The selected rules therefore produce a much smaller list with a higher labelled proportion, but miss 91.51% of the period's laundering labels.

The currency and value condition applies only to Saudi Riyals. Rapid transfer matching applies across currencies, with each amount comparison made within the same currency. Higher precision does not imply broad coverage. The Saudi Riyal condition contributes a substantial share of the result and should not be generalised to other currencies or real transactions. The daily rank orders review workload; it has not been shown to rank actual account risk.

Spark is suitable for this workload because the query processes 5,078,345 transactions and uses 518,581 account records for enrichment. Transfer matching creates incoming and outgoing events, groups them by account and currency, and sorts them by time. Further aggregations and ranking require data to be redistributed between partitions. Spark can execute work across these partitions in parallel and spread the processing and memory requirements across workers as the data grows. This notebook uses `local[*]` to run parallel tasks on one computer; the dataset size alone does not establish that a separate cluster is necessary. The query combines joins, multiple aggregation levels, window functions and time based analysis.

Projection keeps bank and entity text out of the event window. Self transfers and nonpositive amounts are removed before matching, but payment format is checked after choosing the latest receipt: filtering to ACH too early could incorrectly skip a nearer non ACH receipt. Labels are retained for evaluation and never filter transactions before the baseline denominator is calculated.

Aggregation and flag deduplication reduce rows before the enrichment join. The measured flag and daily account results are small enough to broadcast, while the complete account reference need not be broadcast. The event window, distinct counts, grouping and priority ranking can require shuffles. Part B will inspect the actual physical plan and measure those costs.

The DataFrame functions make the event transformations explicit and reusable. SQL expresses equivalent stages as CTEs and makes the window and grouping conditions easy to inspect together. Both use the same source identifiers and parameters but independently construct the matching and aggregation logic. The full result comparison establishes equivalence on this dataset, with tolerance only for monetary sums.

The prepared source is persisted to stabilise identifiers and because both implementations and evaluation reuse it. The small flag and final result caches avoid recomputing windows for every preview and validation action. They are released below. Part B must run complete query functions with equivalent input preparation and cache conditions; these validation runs are not benchmark evidence.

In [18]:
review_flags.unpersist()
dataframe_result.unpersist()
sql_result.unpersist()
transaction_input.unpersist()
print("Part A checks completed; cached inputs and results released.")

Part A checks completed; cached inputs and results released.


## 7. Partitioning strategy

### 7.1 Compare hash and range partitioning

Use `sender_account`, the originating account column selected in the approved proposal. Both strategies process all 5,078,345 transactions and use 28 partitions. The recorded local environment has 14 processing threads, so 28 partitions allow approximately two tasks per thread during the partitioned stage. This provides some scheduling flexibility without creating hundreds of small tasks. It is a practical starting point, not a measured optimum.

Hash partitioning assigns equal account identifiers to the same partition. Range partitioning assigns ranges of account identifiers to partitions using sampled boundaries; it does not guarantee sorted rows within each partition. Because account identifiers are strings, their range order is lexical and does not represent time or risk. Bank remains part of the account identity in the business query; this experiment uses the single column specified in the proposal.

Capture each row's partition identifier before aggregating the counts. Include empty partitions as zero and check that each strategy retains the full source count. Range boundaries and counts can vary between executions because they are sampled.

In [19]:
partition_count = 28
print(f"Processing threads: {sc.defaultParallelism}; partitions per strategy: {partition_count}")

hash_partitioned = transactions.repartition(partition_count, "sender_account")
range_partitioned = transactions.repartitionByRange(partition_count, "sender_account")

partition_sizes = {}
for strategy, partitioned in [("Hash", hash_partitioned), ("Range", range_partitioned)]:
    observed = (
        partitioned.withColumn("partition_id", F.spark_partition_id())
        .groupBy("partition_id").agg(F.count("*").alias("records"))
        .collect()
    )
    counts = {row.partition_id: row.records for row in observed}
    assert all(0 <= key < partition_count for key in counts)
    partition_sizes[strategy] = [counts.get(i, 0) for i in range(partition_count)]
    assert sum(partition_sizes[strategy]) == transaction_count

partition_distribution = spark.createDataFrame(
    [(i, partition_sizes["Hash"][i], partition_sizes["Range"][i])
     for i in range(partition_count)],
    "partition_id int, hash_records long, range_records long"
)
partition_distribution.orderBy("partition_id").show(partition_count, truncate=False)

Processing threads: 14; partitions per strategy: 28


+------------+------------+-------------+
|partition_id|hash_records|range_records|
+------------+------------+-------------+
|0           |190734      |271690       |
|1           |164849      |91283        |
|2           |164286      |158751       |
|3           |164986      |182545       |
|4           |160450      |190646       |
|5           |174190      |159195       |
|6           |164754      |187729       |
|7           |163021      |177299       |
|8           |167392      |172770       |
|9           |162600      |171350       |
|10          |166594      |172297       |
|11          |187788      |183565       |
|12          |165784      |174664       |
|13          |173676      |190789       |
|14          |169472      |183124       |
|15          |180279      |193693       |
|16          |197764      |182188       |
|17          |179644      |183344       |
|18          |167306      |176646       |
|19          |161417      |179646       |
|20          |266663      |196307 

### 7.2 Distribution summary

Compare the smallest and largest partitions, the largest count relative to the mean, and the coefficient of variation. The coefficient of variation is the population standard deviation divided by the mean, expressed as a percentage. Lower values indicate more even row counts. These measures describe partition sizes, not task duration or query speed.

In [20]:
from statistics import mean, pstdev

partition_statistics = []
for strategy, counts in partition_sizes.items():
    average = mean(counts)
    partition_statistics.append((
        strategy, len(counts), sum(counts), min(counts), max(counts),
        average, max(counts) / average, pstdev(counts) / average * 100
    ))
partition_summary = spark.createDataFrame(
    partition_statistics,
    "strategy string, partitions int, total_records long, min_records long, "
    "max_records long, mean_records double, max_to_mean double, cv_percent double"
)
partition_summary.select(
    "strategy", "partitions", "total_records", "min_records", "max_records",
    F.round("mean_records", 2).alias("mean_records"),
    F.round("max_to_mean", 3).alias("max_to_mean"),
    F.round("cv_percent", 2).alias("cv_percent")
).orderBy("strategy").show(truncate=False)

+--------+----------+-------------+-----------+-----------+------------+-----------+----------+
|strategy|partitions|total_records|min_records|max_records|mean_records|max_to_mean|cv_percent|
+--------+----------+-------------+-----------+-----------+------------+-----------+----------+
|Hash    |28        |5078345      |160450     |352326     |181369.46   |1.943      |21.26     |
|Range   |28        |5078345      |91283      |271690     |181369.46   |1.498      |14.64     |
+--------+----------+-------------+-----------+-----------+------------+-----------+----------+



### 7.3 Partitioning discussion

Both strategies retain all 5,078,345 transactions across 28 nonempty partitions, giving a mean of 181,369.46 records per partition. Hash partitioning produces mostly similar counts but has two larger partitions: partition 20 contains 266,663 rows and partition 26 contains 352,326. Its largest partition is 1.943 times the mean, and its coefficient of variation is 21.26%. These counts show uneven work distribution that could leave some tasks running longer than others.

Range partitioning is more balanced overall in this run, with a coefficient of variation of 14.64% and a largest partition of 271,690 rows, or 1.498 times the mean. However, its smallest partition contains only 91,283 rows, so it does not eliminate imbalance. Sampled boundaries and repeated account identifiers can affect range sizes; both strategies keep identical partition keys together. The measured counts establish imbalance but do not by themselves identify its cause.

Hash partitioning suits grouping and equality joins on sender account. Range partitioning may suit processing by account ranges, although these identifiers have no chronological meaning. Neither layout directly matches the full Part A event window, which uses period, bank, account and currency, including receiving accounts. Spark may therefore introduce further exchanges and sorting. Both repartitioning operations also incur a shuffle themselves. The more even range counts suggest potential scheduling benefits, but they do not prove lower query execution time. Timing and execution plan evidence are needed before recommending either strategy for the complete query.

## 8. Execution time benchmarking

### 8.1 Benchmark conditions

Measure the complete Part A query from the prepared transaction input through transfer matching, flag deduplication, daily aggregation, ranking, account enrichment and collection of every result row. Each timed cell constructs a fresh query and calls `collect()`; it does not reuse the Part A flags or final results. The account reference is read from its CSV during query execution.

Prepare and materialise the common transaction input once outside the timings. This gives both APIs the same cached input and stable source row identifiers. CSV loading and identifier preparation for this input are excluded from both timings. Intermediate flags and query results are not cached during benchmarking. The hash and range layouts in Section 7 are separate experiments and are not used here.

Run DataFrame and SQL alternately for three runs each in the same local Spark session. Earlier notebook cells have already warmed the JVM, so these measure repeated queries rather than a fresh application startup. Keep Spark settings unchanged between runs. `%%time` records each cell's wall time, while `perf_counter()` records the query construction and collection interval in seconds for the comparison table. Cell wall time also includes the small amount of timing bookkeeping and printing. These are single computer measurements, not cluster results.

In [21]:
from time import perf_counter
from statistics import median

for cached_frame in [review_flags, dataframe_result, sql_result, transaction_input]:
    cached_frame.unpersist(blocking=True)
transaction_input.persist(StorageLevel.MEMORY_AND_DISK)
assert transaction_input.count() == transaction_count
transaction_input.createOrReplaceTempView("transaction_input")
accounts.createOrReplaceTempView("account_reference")

benchmark_timings = []
benchmark_results = {}
print("Spark version:", spark.version)
print("Execution mode:", sc.master)
print("Processing threads:", sc.defaultParallelism)
print("Driver memory:", sc.getConf().get("spark.driver.memory"))
print("Executor memory: shared driver JVM in local mode; no separate executor heap")
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Adaptive query execution:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Input: prepared transactions cached; account CSV uncached; query results uncached")

Spark version: 3.5.7
Execution mode: local[*]
Processing threads: 14
Driver memory: 4g
Executor memory: shared driver JVM in local mode; no separate executor heap
Shuffle partitions: 200
Adaptive query execution: true
Input: prepared transactions cached; account CSV uncached; query results uncached


### 8.2 DataFrame run 1

In [22]:
%%time
benchmark_start = perf_counter()
benchmark_query = build_dataframe_query(transaction_input, accounts)
benchmark_rows = benchmark_query.collect()
benchmark_seconds = perf_counter() - benchmark_start
benchmark_timings.append(("DataFrame", 1, benchmark_seconds))
benchmark_results[("DataFrame", 1)] = benchmark_rows
print(f"DataFrame run 1: {len(benchmark_rows):,} rows in {benchmark_seconds:.3f} seconds")

DataFrame run 1: 3,976 rows in 11.453 seconds
CPU times: user 35 ms, sys: 22.1 ms, total: 57.1 ms
Wall time: 11.5 s


### 8.3 Spark SQL run 1

In [23]:
%%time
benchmark_start = perf_counter()
benchmark_query = build_sql_query()
benchmark_rows = benchmark_query.collect()
benchmark_seconds = perf_counter() - benchmark_start
benchmark_timings.append(("Spark SQL", 1, benchmark_seconds))
benchmark_results[("Spark SQL", 1)] = benchmark_rows
print(f"Spark SQL run 1: {len(benchmark_rows):,} rows in {benchmark_seconds:.3f} seconds")

Spark SQL run 1: 3,976 rows in 19.293 seconds
CPU times: user 33.2 ms, sys: 18.1 ms, total: 51.3 ms
Wall time: 19.3 s


### 8.4 DataFrame run 2

In [24]:
%%time
benchmark_start = perf_counter()
benchmark_query = build_dataframe_query(transaction_input, accounts)
benchmark_rows = benchmark_query.collect()
benchmark_seconds = perf_counter() - benchmark_start
benchmark_timings.append(("DataFrame", 2, benchmark_seconds))
benchmark_results[("DataFrame", 2)] = benchmark_rows
print(f"DataFrame run 2: {len(benchmark_rows):,} rows in {benchmark_seconds:.3f} seconds")

DataFrame run 2: 3,976 rows in 10.289 seconds
CPU times: user 30.6 ms, sys: 20.9 ms, total: 51.6 ms
Wall time: 10.3 s


### 8.5 Spark SQL run 2

In [25]:
%%time
benchmark_start = perf_counter()
benchmark_query = build_sql_query()
benchmark_rows = benchmark_query.collect()
benchmark_seconds = perf_counter() - benchmark_start
benchmark_timings.append(("Spark SQL", 2, benchmark_seconds))
benchmark_results[("Spark SQL", 2)] = benchmark_rows
print(f"Spark SQL run 2: {len(benchmark_rows):,} rows in {benchmark_seconds:.3f} seconds")

Spark SQL run 2: 3,976 rows in 19.801 seconds
CPU times: user 38.9 ms, sys: 21.2 ms, total: 60.1 ms
Wall time: 19.8 s


### 8.6 DataFrame run 3

In [26]:
%%time
benchmark_start = perf_counter()
benchmark_query = build_dataframe_query(transaction_input, accounts)
benchmark_rows = benchmark_query.collect()
benchmark_seconds = perf_counter() - benchmark_start
benchmark_timings.append(("DataFrame", 3, benchmark_seconds))
benchmark_results[("DataFrame", 3)] = benchmark_rows
print(f"DataFrame run 3: {len(benchmark_rows):,} rows in {benchmark_seconds:.3f} seconds")

DataFrame run 3: 3,976 rows in 10.671 seconds
CPU times: user 32 ms, sys: 21.3 ms, total: 53.3 ms
Wall time: 10.7 s


### 8.7 Spark SQL run 3

In [27]:
%%time
benchmark_start = perf_counter()
benchmark_query = build_sql_query()
benchmark_rows = benchmark_query.collect()
benchmark_seconds = perf_counter() - benchmark_start
benchmark_timings.append(("Spark SQL", 3, benchmark_seconds))
benchmark_results[("Spark SQL", 3)] = benchmark_rows
print(f"Spark SQL run 3: {len(benchmark_rows):,} rows in {benchmark_seconds:.3f} seconds")

Spark SQL run 3: 3,976 rows in 19.815 seconds
CPU times: user 56.1 ms, sys: 28.8 ms, total: 84.8 ms
Wall time: 19.8 s


### 8.8 Timing comparison

Report all three runs and the median in seconds. Check that every run returns the same complete result as the first DataFrame run, allowing the monetary tolerance already used in Part A. Perform these checks outside the timed cells.

In [28]:
assert len(benchmark_timings) == 6
assert len(benchmark_results) == 6
reference_rows = {
    tuple(row[key] for key in result_keys): row.asDict()
    for row in benchmark_results[("DataFrame", 1)]
}
assert len(reference_rows) == dataframe_result_count
for (api, run), rows in benchmark_results.items():
    assert len(rows) == dataframe_result_count
    actual_rows = {tuple(row[key] for key in result_keys): row.asDict() for row in rows}
    assert actual_rows.keys() == reference_rows.keys()
    for key, actual in actual_rows.items():
        expected = reference_rows[key]
        assert {c: v for c, v in actual.items() if c != "flagged_amount_paid"} == {
            c: v for c, v in expected.items() if c != "flagged_amount_paid"
        }
        assert abs(actual["flagged_amount_paid"] - expected["flagged_amount_paid"]) <= max(
            1e-8, abs(expected["flagged_amount_paid"]) * 1e-10
        )

timing_comparison_rows = []
for api in ["DataFrame", "Spark SQL"]:
    runs = [seconds for name, run, seconds in sorted(benchmark_timings, key=lambda item: item[1])
            if name == api]
    timing_comparison_rows.append((api, *runs, median(runs)))
timing_comparison = spark.createDataFrame(
    timing_comparison_rows,
    "API string, run_1_seconds double, run_2_seconds double, run_3_seconds double, median_seconds double"
)
timing_comparison.select(
    "API", *[F.round(c, 3).alias(c) for c in timing_comparison.columns if c != "API"]
).orderBy("API").show(truncate=False)
print(f"All six runs return the same {len(reference_rows):,} result rows within monetary tolerance.")
transaction_input.unpersist(blocking=True)

+---------+-------------+-------------+-------------+--------------+
|API      |run_1_seconds|run_2_seconds|run_3_seconds|median_seconds|
+---------+-------------+-------------+-------------+--------------+
|DataFrame|11.453       |10.289       |10.671       |10.671        |
|Spark SQL|19.293       |19.801       |19.815       |19.801        |
+---------+-------------+-------------+-------------+--------------+

All six runs return the same 3,976 result rows within monetary tolerance.


DataFrame[timestamp: timestamp, from_bank: bigint, sender_account: string, to_bank: bigint, receiver_account: string, amount_received: double, receiving_currency: string, amount_paid: double, payment_currency: string, payment_format: string, is_laundering: int, row_id: bigint]

### 8.9 Benchmark discussion

DataFrame recorded a median of 10.671 seconds, compared with 19.801 seconds for Spark SQL. Its median elapsed time was about 46.1% lower. DataFrame runs ranged from 10.289 to 11.453 seconds, while SQL ranged from 19.293 to 19.815 seconds. Every run returned the same 3,976 rows within the monetary tolerance, so the difference is not explained by one implementation producing fewer results.

Both APIs use Spark SQL's Catalyst optimiser and execute distributed operators in the JVM. DataFrame syntax is therefore not inherently faster than SQL. These implementations express equivalent logic differently: DataFrame expands the two transfer legs with an array and explode, while SQL references matched transfers in separate UNION ALL branches. The DataFrame code also explicitly broadcasts the flag relation. These choices can affect exchange reuse, join selection and intermediate work. The execution plan analysis is needed to establish which differences actually occurred; the timings alone cannot identify the cause.

The event window requires grouping and sorting, while aggregation and ranking can add shuffles. On this local machine, these involve memory, disk and scheduling costs rather than communication between separate worker machines. Collection also includes conversion and transfer of the final rows to Python for both APIs.

The benchmark used 14 processing threads, 4 GB driver memory and identical cached transaction input. Earlier cells had already warmed the JVM. Alternating APIs and reporting medians limits dependence on one run, but three runs cannot remove effects from garbage collection, operating system caching or other local activity. These measurements support the DataFrame implementation in this environment, not a general API performance claim.

## 9. Execution plan interpretation

### 9.1 Explain the flag aggregation

Select the final aggregation in `build_dataframe_flags()` from Part A. It groups by source row identifier and takes the maximum of each reason flag, ensuring that a transaction appears once even when several conditions or transfer pairs match. The plan includes the preceding transfer window because Spark evaluates the aggregation's full dependency chain.

Recreate the shared prepared input and leave the flag result uncached. Increase the plan display limits while printing so long expressions are retained. The complete extended output below includes the parsed, analysed and optimised logical plans and the physical plan before execution. Adaptive Query Execution may revise the physical plan when the action in Section 10 runs. The output is saved in full; Spark itself abbreviates the cached CSV scan metadata named `ReadSchema`, whose complete schema is already printed in Section 2.

In [31]:
transaction_input.persist(StorageLevel.MEMORY_AND_DISK)
assert transaction_input.count() == transaction_count
plan_flags = build_dataframe_flags(transaction_input)
plan_display_settings = {
    "spark.sql.debug.maxToStringFields": "1000",
    "spark.sql.maxMetadataStringLength": "10000"
}
previous_plan_settings = {key: spark.conf.get(key) for key in plan_display_settings}
try:
    for key, value in plan_display_settings.items():
        spark.conf.set(key, value)
    plan_flags.explain(extended=True)
finally:
    for key, value in previous_plan_settings.items():
        spark.conf.set(key, value)

== Parsed Logical Plan ==
'Aggregate ['row_id], ['row_id, max('value_flag) AS value_flag#44619, max('transfer_flag) AS transfer_flag#44621]
+- Union false, false
   :- Project [row_id#601L, 1 AS value_flag#44595, 0 AS transfer_flag#44596]
   :  +- Filter ((((payment_format#9 = ACH) AND (payment_currency#8 = Saudi Riyal)) AND (amount_paid#7 >= 10000.0)) AND (amount_paid#7 < 100000.0))
   :     +- Project [timestamp#0, from_bank#1L, sender_account#2, to_bank#3L, receiver_account#4, amount_received#5, receiving_currency#6, amount_paid#7, payment_currency#8, payment_format#9, is_laundering#10, monotonically_increasing_id() AS row_id#601L]
   :        +- Relation [timestamp#0,from_bank#1L,sender_account#2,to_bank#3L,receiver_account#4,amount_received#5,receiving_currency#6,amount_paid#7,payment_currency#8,payment_format#9,is_laundering#10] csv
   +- Project [row_id#44591L, value_flag#44588, transfer_flag#44589]
      +- Project [row_id#44591L, 0 AS value_flag#44588, 1 AS transfer_flag#44589

### 9.2 Annotated physical plan

These excerpts retain the operator text from the physical plan above and add inline comments. Read each branch from its child upwards: the operator below an Exchange supplies its input.

**Flag deduplication aggregation**

```text
+- HashAggregate(keys=[row_id#601L], functions=[max(value_flag#44595), max(transfer_flag#44596)], output=[row_id#601L, value_flag#44619, transfer_flag#44621])  # Final aggregation merges flags for each source row
   +- Exchange hashpartitioning(row_id#601L, 200), ENSURE_REQUIREMENTS, [plan_id=38883]  # Shuffle 2: bring equal row identifiers together
      +- HashAggregate(keys=[row_id#601L], functions=[partial_max(value_flag#44595), partial_max(transfer_flag#44596)], output=[row_id#601L, max#45169, max#45170])  # Immediate input to Shuffle 2: partial HashAggregate
```

**Transfer window preparation**

```text
                           +- Sort [review_period#44470 ASC NULLS FIRST, bank#44516L ASC NULLS FIRST, account#44517 ASC NULLS FIRST, currency#44520 ASC NULLS FIRST, event_second#44483L ASC NULLS FIRST], false, 0  # Sort the received event groups before the window
                              +- Exchange hashpartitioning(review_period#44470, bank#44516L, account#44517, currency#44520, 200), ENSURE_REQUIREMENTS, [plan_id=38873]  # Shuffle 1: place each window group in one partition
                                 +- Union  # Immediate input to Shuffle 1: Union of incoming and outgoing events
```

The 200 partitions are the initial shuffle setting, distinct from the 28 partitions deliberately requested in Section 7. The actual execution in Section 10 shows how AQE coalesces the shuffle reads.

### 9.3 Shuffle analysis

**Why the exchanges occur.** The first exchange groups incoming and outgoing events by period, bank, account and currency. Its immediate input is Union. Events for an account can originate in different source partitions, so Spark must bring each window group together before sorting by timestamp and finding the latest receipt. The second exchange groups candidate flags by source row identifier. Its immediate input is a partial HashAggregate, which calculates local maximum flags before the final HashAggregate merges contributions from all partitions. Local aggregation alone cannot guarantee one result per transaction.

**Execution cost.** Each shuffle writes partitioned intermediate data and creates a dependency between producing and consuming stages. Consumers must fetch the completed shuffle blocks, adding serialisation, disk access, memory pressure and task scheduling overhead. On a cluster, remote fetches also use network bandwidth. This local run reports zero remote shuffle bytes, so its measured costs should not be described as transfers between separate machines. Adaptive execution can reduce the number of downstream tasks without removing the underlying shuffle requirement.

**Possible reductions.** The query already projects the needed event fields, removes self transfers and nonpositive amounts, and partially aggregates flags before redistribution. Filtering to ACH before selecting the nearest receipt would change the business rule. Reusing event data partitioned by the complete window key could avoid a repeated exchange, but preparing that layout has its own cost. Partitioning only by sender account does not match both event directions or the complete key. A broadcast join can help later account enrichment; it cannot replace the grouping required by these two exchanges.

## 10. Spark Web UI DAG analysis

### 10.1 Execute the selected Part A operation

Execute the same flag derivation used in Section 9 and collect all flag rows. This is the transfer matching and deduplication portion of the Part A DataFrame implementation, before daily ranking and account enrichment. Open the printed Spark Web UI address while the Spark session remains active, select the SQL execution associated with this action and inspect its DAG. The screenshot records this execution; application and stage identifiers may differ when the notebook is rerun.

In [32]:
sc.setJobGroup("part-b-flag-dag", "Part B: transfer matching and flag deduplication")
try:
    dag_flag_rows = plan_flags.collect()
    assert len(dag_flag_rows) == flagged_transaction_count
    print(f"Collected {len(dag_flag_rows):,} unique flagged transactions.")
    print("Spark application:", sc.applicationId)
    print("Spark Web UI:", sc.uiWebUrl)
finally:
    sc.setLocalProperty("spark.jobGroup.id", None)
    sc.setLocalProperty("spark.job.description", None)
    sc.setLocalProperty("spark.job.interruptOnCancel", None)
    transaction_input.unpersist(blocking=True)

Collected 4,159 unique flagged transactions.
Spark application: local-1790990936040
Spark Web UI: http://localhost:4041


### 10.2 DAG evidence

Captured from the Spark Web UI SQL / DataFrame tab for application `local-1790990936040`, query **58**, submitted on **3 October 2026 at 11:34:06**. It completed in approximately six seconds. Stage identifiers are enabled in the screenshot, including stages **594** and **596**. The linked job details identify **599** as the final result stage. WholeStageCodegen numbers identify generated operator pipelines and are not Spark stage IDs.

![DAG Screenshot](./dag.png)

| Spark stage | Observed work | Evidence |
| --- | --- | --- |
| 594 | Read cached input, construct incoming and outgoing events, and write the event shuffle | 8,974,266 event rows; 574.5 MiB shuffled |
| 596 | Read event groups, sort and match transfers, combine both rule branches, and partially aggregate flags | 968 matched transfers; 4,211 partial flag rows; 218.7 KiB written to the second shuffle |
| 599 | Read the coalesced flag shuffle and perform final aggregation | 4,159 unique flagged transactions |

The two exchanges initially specify 200 partitions. AQE reads the event shuffle through 15 coalesced partitions and the final flag shuffle through one. Stage 596 also processes the separate currency condition branch, so its task workload is not limited to the 15 event partitions.

### 10.3 DAG interpretation

The exchanges create the stage boundaries. Stage 594 writes events grouped by the window key; stage 596 consumes those events for sorting and transfer matching. After combining the transfer and currency conditions, a partial HashAggregate writes flags grouped by source row identifier. Stage 599 merges these into the final 4,159 rows. Narrow operations such as projection and filtering can execute within a stage without another redistribution.

The event shuffle is the larger operation: 8,974,266 rows and 574.5 MiB, compared with 4,211 rows and 218.7 KiB for the flag shuffle. Both report zero remote bytes read, consistent with local execution. The Sort operator reports 1,664 MiB in its spill metric, showing memory pressure during sorting; this operator metric should not be interpreted as the exact number of disk bytes written.

The coalesced event partitions range from 24.8 to 42.5 MiB, with a median of 41.3 MiB. The largest is close to the median, so this view does not show an extreme oversized event partition. Processing durations vary, but differing branch work and spilling mean that duration variation alone does not prove account key skew.

Possible improvements include testing smaller AQE target partitions to reduce memory per sorting task, balancing that against extra scheduling overhead, and reducing repeated event preparation where reuse is possible. More available execution memory may reduce spill. These are candidates for measurement, not demonstrated improvements. The two required grouping operations cannot simply be removed, and the small final shuffle is a lower priority than event sorting and redistribution.